In [1]:
# importing libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score ,
    precision_score ,
    recall_score ,
    f1_score ,
    average_precision_score , 
    roc_auc_score ,
    confusion_matrix ,
    classification_report, 
    ConfusionMatrixDisplay 
)

In [2]:
catboost_oof = pd.read_csv("../outputs/oof/catboost_oof.csv")

xgb_123 = pd.read_csv("../outputs/oof/xgboost_seed_123_oof.csv")
xgb_2024 = pd.read_csv("../outputs/oof/xgboost_seed_2024_oof.csv")
xgb_777 = pd.read_csv("../outputs/oof/xgboost_seed_777_oof.csv")

lgbm_123 = pd.read_csv("../outputs/oof/lightgbm_seed_123_oof.csv")
lgbm_2024 = pd.read_csv("../outputs/oof/lightgbm_seed_2024_oof.csv")
lgbm_777 = pd.read_csv("../outputs/oof/lightgbm_seed_777_oof.csv")

In [3]:
lgbm_123

,isFraud,lightbgm_seed_123_oof
0,0,NaN
1,0,NaN
2,0,NaN
3,0,NaN
4,0,NaN
...,...,...
590535,0,0.008005
590536,0,0.010325
590537,0,0.004737
590538,0,0.063647


In [4]:
xgb_seed_ensemble = (xgb_123["xgboost_seed_123_oof"] + xgb_2024["xgboost_seed_2024_oof"] + xgb_777["xgboost_seed_777_oof"])/3
lgbm_seed_ensemble = (lgbm_123["lightbgm_seed_123_oof"] + lgbm_2024["lightbgm_seed_2024_oof"] + lgbm_777["lightbgm_seed_777_oof"])/3

In [5]:
np.array_equal(catboost_oof["isFraud"].values , xgb_123["isFraud"].values) , np.array_equal(catboost_oof["isFraud"].values , lgbm_123["isFraud"].values)

(True, True)

In [6]:
catboost_oof["catboost_oof"].isna().sum() , xgb_seed_ensemble.isna().sum() , lgbm_seed_ensemble.isna().sum()

(np.int64(98425), np.int64(98425), np.int64(98425))

In [7]:
stack_df = pd.DataFrame({
    "isFraud" : catboost_oof["isFraud"] , 
    "catboost" : catboost_oof["catboost_oof"] ,
    "lightgbm" : lgbm_seed_ensemble ,
    "xgboost" : xgb_seed_ensemble
})

In [8]:
stack_df = stack_df.dropna().reset_index(drop=True)

In [9]:
x_meta = stack_df[["catboost" , "lightgbm" , "xgboost"]]
y_meta = stack_df["isFraud"]

In [10]:
from sklearn.model_selection import TimeSeriesSplit

tscv_meta = TimeSeriesSplit(n_splits=5 , gap=0)

In [11]:
lr_oof = np.full(len(x_meta) , np.nan)
for fold , (train_idx , val_idx) in enumerate(tscv_meta.split(x_meta) , start=1):
    x_train = x_meta.iloc[train_idx].copy()
    x_val = x_meta.iloc[val_idx].copy()

    y_train = y_meta.iloc[train_idx].copy()
    y_val = y_meta.iloc[val_idx].copy()

    lr_model = Pipeline([
        ("scaler" , StandardScaler()) ,
        ("model" , LogisticRegression(max_iter=1000))
    ])

    lr_model.fit(x_train , y_train)
    val_prob = lr_model.predict_proba(x_val)[:,1]

    lr_oof[val_idx] = val_prob
    fold_roc_auc = roc_auc_score(y_val , val_prob)
    fold_pr_auc = average_precision_score(y_val , val_prob)
    
    print(f"Fold {fold} | " 
        f"ROC-AUC : {fold_roc_auc:.4f} | "
        f"PR-AUC : {fold_pr_auc:.4f}")
    
lr_mask = ~np.isnan(lr_oof)    
lr_roc_auc = roc_auc_score(y_meta.iloc[lr_mask] , lr_oof[lr_mask])
lr_pr_auc = average_precision_score(y_meta.iloc[lr_mask] , lr_oof[lr_mask])

print(f"Logistic Regression Meta OOF ROC-AUC : {lr_roc_auc:.4f}")
print(f"Logistic Regression Meta OOF PR-AUC : {lr_pr_auc:.4f}")

Fold 1 | ROC-AUC : 0.9166 | PR-AUC : 0.6253
Fold 2 | ROC-AUC : 0.9151 | PR-AUC : 0.6034
Fold 3 | ROC-AUC : 0.9319 | PR-AUC : 0.6556
Fold 4 | ROC-AUC : 0.9342 | PR-AUC : 0.6248
Fold 5 | ROC-AUC : 0.9327 | PR-AUC : 0.6180
Logistic Regression Meta OOF ROC-AUC : 0.9197
Logistic Regression Meta OOF PR-AUC : 0.6252


In [12]:
rf_oof = np.full(len(x_meta) , np.nan)
for fold , (train_idx , val_idx) in enumerate(tscv_meta.split(x_meta) , start=1):
    x_train = x_meta.iloc[train_idx].copy()
    x_val = x_meta.iloc[val_idx].copy()

    y_train = y_meta.iloc[train_idx].copy()
    y_val = y_meta.iloc[val_idx].copy()

    rf_model = RandomForestClassifier(n_estimators=200 , max_depth=5 , random_state=42 , n_jobs=-1)

    rf_model.fit(x_train , y_train)
    val_prob = rf_model.predict_proba(x_val)[:,1]

    rf_oof[val_idx] = val_prob
    fold_roc_auc = roc_auc_score(y_val , val_prob)
    fold_pr_auc = average_precision_score(y_val , val_prob)
    
    print(f"Fold {fold} | " 
        f"ROC-AUC : {fold_roc_auc:.4f} | "
        f"PR-AUC : {fold_pr_auc:.4f}")
    
rf_mask = ~np.isnan(rf_oof)    
rf_roc_auc = roc_auc_score(y_meta.iloc[rf_mask] , rf_oof[rf_mask])
rf_pr_auc = average_precision_score(y_meta.iloc[rf_mask] , rf_oof[rf_mask])

print(f"Random Forest Meta OOF ROC-AUC : {rf_roc_auc:.4f}")
print(f"Random Forest Meta OOF PR-AUC : {rf_pr_auc:.4f}")    

Fold 1 | ROC-AUC : 0.9223 | PR-AUC : 0.6419
Fold 2 | ROC-AUC : 0.9164 | PR-AUC : 0.6099
Fold 3 | ROC-AUC : 0.9311 | PR-AUC : 0.6582
Fold 4 | ROC-AUC : 0.9335 | PR-AUC : 0.6287
Fold 5 | ROC-AUC : 0.9315 | PR-AUC : 0.6198
Random Forest Meta OOF ROC-AUC : 0.9266
Random Forest Meta OOF PR-AUC : 0.6330


In [13]:
knn_oof = np.full(len(x_meta) , np.nan)
for fold , (train_idx , val_idx) in enumerate(tscv_meta.split(x_meta) , start=1):
    x_train = x_meta.iloc[train_idx].copy()
    x_val = x_meta.iloc[val_idx].copy()

    y_train = y_meta.iloc[train_idx].copy()
    y_val = y_meta.iloc[val_idx].copy()

    knn_model = Pipeline([
        ("scaler" , StandardScaler()) ,
        ("model" , KNeighborsClassifier(n_neighbors=5))
    ])

    knn_model.fit(x_train , y_train)
    val_prob = knn_model.predict_proba(x_val)[:,1]

    knn_oof[val_idx] = val_prob
    fold_roc_auc = roc_auc_score(y_val , val_prob)
    fold_pr_auc = average_precision_score(y_val , val_prob)
    
    print(f"Fold {fold} | " 
        f"ROC-AUC : {fold_roc_auc:.4f} | "
        f"PR-AUC : {fold_pr_auc:.4f}")
    
knn_mask = ~np.isnan(knn_oof)    
knn_roc_auc = roc_auc_score(y_meta.iloc[knn_mask] , knn_oof[knn_mask])
knn_pr_auc = average_precision_score(y_meta.iloc[knn_mask] , knn_oof[knn_mask])

print(f"KNN Meta OOF ROC-AUC : {knn_roc_auc:.4f}")
print(f"KNN Forest Meta OOF PR-AUC : {knn_pr_auc:.4f}")    

Fold 1 | ROC-AUC : 0.8275 | PR-AUC : 0.4980
Fold 2 | ROC-AUC : 0.8095 | PR-AUC : 0.4707
Fold 3 | ROC-AUC : 0.8301 | PR-AUC : 0.5171
Fold 4 | ROC-AUC : 0.8311 | PR-AUC : 0.4857
Fold 5 | ROC-AUC : 0.8200 | PR-AUC : 0.4695
KNN Meta OOF ROC-AUC : 0.8239
KNN Forest Meta OOF PR-AUC : 0.4892


In [14]:
stacking_oof_df = pd.DataFrame({
    "isFraud" : y_meta.values ,
    "logistic_regression_oof" : lr_oof , 
    "random_forest_oof" : rf_oof ,
    "knn_oof" : knn_oof
})

stacking_oof_df.to_csv("../outputs/oof/stacking_meta_oof.csv" , index=False)